In [1]:
import pandas as pd
import re
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from urllib.parse import urlparse

import nltk
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [2]:
data = pd.read_csv('Data.csv')
data.head()

,Data
0,Watch or listen live weekdays at 8:30am MT at ...
1,Watch or listen live weekdays at 8:30am MT at ...
2,"Chubby And Hot, Always Stir The Pot!"
3,"Chubby And Hot, Always Stir The Pot!"
4,"Journalist, publisher of Rebel News — telling ..."


In [3]:
pattern_date = r'\b\d{2,4}[/-]\d{1,2}[/-]\d{1,2}\b'
date_matches = data['Data'].str.findall(pattern_date).dropna()
date_count = sum(date_matches.map(len))
print(f"Number of records with dates without alphabets: {date_count}")

Number of records with dates without alphabets: 0


In [4]:
pattern_w = r'\bw\w*'
w_matches = data['Data'].str.findall(pattern_w, flags=re.IGNORECASE).dropna()
w_count = sum(w_matches.map(len))
print(f"Number of words starting with 'w': {w_count}")

Number of words starting with 'w': 5584


In [5]:
pattern_alpha = r'\b[a-zA-Z][a-zA-Z0-9]*\b'
pattern_url = r'https?://\S+'
alpha_matches = data['Data'].str.findall(pattern_alpha).dropna()
url_matches = data['Data'].str.findall(pattern_url).dropna()
alpha_count = sum(alpha_matches.map(len)) - sum(url_matches.map(len))
print(f"Number of words starting with an alphabet and not URLs: {alpha_count}")

Number of words starting with an alphabet and not URLs: 119159


In [6]:
pattern_emoji = r':\)|:D|;\)|:P'
emoji_matches = data['Data'].str.findall(pattern_emoji).dropna()
emoji_count = sum(emoji_matches.map(len))
print(f"Number of tweets containing specified emojis: {emoji_count}")

Number of tweets containing specified emojis: 28


In [7]:
pattern_decimal = r'\b\d+\.\d+\b'
decimal_matches = data['Data'].str.findall(pattern_decimal).dropna()
decimal_count = sum(decimal_matches.map(len))
print(f"Number of records containing decimal numbers: {decimal_count}")

Number of records containing decimal numbers: 36


In [8]:
pattern_ip = r'\b(?:\d{1,3}\.){3}\d{1,3}\b'
ip_matches = data['Data'].str.findall(pattern_ip).dropna()
ip_count = sum(ip_matches.map(len))
print(f"Total number of IP addresses: {ip_count}")

Total number of IP addresses: 0


In [9]:
newline_matches = data['Data'].str.contains('\n', na=False)
newline_count = newline_matches.sum()
print(f"Number of records with a new line: {newline_count}")

Number of records with a new line: 1211


In [10]:
pattern_hashtag = r'#\w+'
hashtag_matches = data['Data'].str.findall(pattern_hashtag).dropna()
hashtag_count = sum(hashtag_matches.map(len))
print(f"Total number of hashtags: {hashtag_count}")

Total number of hashtags: 2924


In [11]:
data['Cleaned_Data'] = data['Data'].str.replace(r'\W+', '\n', regex=True)
print("Substituted all non-alphanumeric characters with a newline.")

Substituted all non-alphanumeric characters with a newline.


In [12]:
pattern_url = r'https?://\S+'
url_matches = data['Data'].str.findall(pattern_url).dropna()
url_count = sum(url_matches.map(len))
print(f"Total number of URLs: {url_count}")

Total number of URLs: 4


In [13]:
ps = PorterStemmer()

def stem_text(text):
    return ' '.join([ps.stem(word) for word in text.split()])

data['stemmed_text'] = [stem_text(text) for text in data['Data']]

unique_words_before_stemming = len(set(' '.join(data['Data']).split()))
unique_words_after_stemming = len(set(' '.join(data['stemmed_text']).split()))

print(f"Unique words before stemming: {unique_words_before_stemming}")
print(f"Unique words after stemming: {unique_words_after_stemming}")

Unique words before stemming: 13043
Unique words after stemming: 10404


In [14]:
lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    return ' '.join([lemmatizer.lemmatize(word) for word in text.split()])

data['lemmatized_text'] = [lemmatize_text(text) for text in data['Data']]

unique_words_before_lemmatization = len(set(' '.join(data['Data']).split()))
unique_words_after_lemmatization = len(set(' '.join(data['lemmatized_text']).split()))

print(f"Unique words before lemmatization: {unique_words_before_lemmatization}")
print(f"Unique words after lemmatization: {unique_words_after_lemmatization}")

Unique words before lemmatization: 13043
Unique words after lemmatization: 12794


In [15]:
from collections import Counter

lemmatized_word_freq = Counter(' '.join(data['lemmatized_text']).split())

lemmatize_df = pd.DataFrame({
    'Top Words After Lemmatization': [word[0] for word in lemmatized_word_freq.most_common(10)],
    'Frequency After Lemmatization': [word[1] for word in lemmatized_word_freq.most_common(10)]
})
lemmatize_df

,Top Words After Lemmatization,Frequency After Lemmatization
0,and,3477
1,the,2472
2,of,2191
3,to,2147
4,&,1639
5,a,1568
6,in,1509
7,for,1380
8,|,1231
9,is,895


In [16]:
stemmed_word_freq = Counter(' '.join(data['stemmed_text']).split())
stem_df = pd.DataFrame({
    'Top Words After Stemming': [word[0] for word in stemmed_word_freq.most_common(10)],
    'Frequency After Stemming': [word[1] for word in stemmed_word_freq.most_common(10)],
})
stem_df

,Top Words After Stemming,Frequency After Stemming
0,and,3508
1,the,2829
2,of,2215
3,to,2167
4,&,1639
5,a,1548
6,in,1531
7,for,1502
8,|,1231
9,is,910


In [17]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word.lower() not in stop_words])

normalized_text = [remove_stopwords(text) for text in data['Data']]
normalized_word_freq = Counter(' '.join(normalized_text).split())

stopwords_df = pd.DataFrame({
    'Top Words After Stop-Word Removal': [word[0] for word in normalized_word_freq.most_common(10)],
    'Frequency': [word[1] for word in normalized_word_freq.most_common(10)]
})
stopwords_df

,Top Words After Stop-Word Removal,Frequency
0,&,1639
1,|,1231
2,Alberta,642
3,-,557
4,Outdoor,451
5,Canada,304
6,news,293
7,Follow,241
8,"Writer,",209
9,online,199
